# 2.4 半监督学习 (Semi-Supervised Learning)

结合少量标注数据和大量未标注数据共同训练，降低对标注数据的依赖。

本节涵盖：
- 自训练
- 伪标签
- 一致性正则化
- 协同训练
- LLM辅助标注

## 1. 自训练 / 2. 伪标签

**目的**：利用模型自身预测扩展标注数据

**基本原理**：先用少量标注数据训练初始模型，再用模型对未标注数据进行预测，将高置信度的预测结果作为伪标签。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SimpleClassifier(nn.Module):
    def __init__(self, dim=10, hidden=32, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x):
        return self.net(x)

def generate_data(n_labeled=30, n_unlabeled=200, dim=10, n_classes=3):
    centers = torch.randn(n_classes, dim) * 3
    labels_l = torch.randint(0, n_classes, (n_labeled,))
    X_l = centers[labels_l] + torch.randn(n_labeled, dim) * 0.5
    X_u = centers[torch.randint(0, n_classes, (n_unlabeled,))] + torch.randn(n_unlabeled, dim) * 0.5
    return X_l, labels_l, X_u

X_l, y_l, X_u = generate_data()
model = SimpleClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print('=== Self-Training with Pseudo-Labels ===')
print(f'Labeled: {X_l.shape[0]}, Unlabeled: {X_u.shape[0]}')

for step in range(5):
    # Step 1: Train on labeled data
    for epoch in range(20):
        logits = model(X_l)
        loss = F.cross_entropy(logits, y_l)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Step 2: Generate pseudo-labels for unlabeled data
    with torch.no_grad():
        logits_u = model(X_u)
        probs = F.softmax(logits_u, dim=-1)
        max_probs, pseudo_labels = probs.max(dim=-1)
    
    # Step 3: Select high-confidence pseudo-labels
    threshold = 0.9
    confident = max_probs > threshold
    n_confident = confident.sum().item()
    
    if n_confident > 0:
        X_pseudo = X_u[confident]
        y_pseudo = pseudo_labels[confident]
        X_l = torch.cat([X_l, X_pseudo])
        y_l = torch.cat([y_l, y_pseudo])
        X_u = X_u[~confident]
    
    acc = (model(X_l[:30]).argmax(dim=-1) == y_l[:30]).float().mean()
    print(f'  Iter {step+1}: labeled={X_l.shape[0]}, unlabeled={X_u.shape[0]}, '
          f'pseudo_added={n_confident}, acc={acc:.3f}')

## 3. 一致性正则化

**目的**：利用模型对扰动的鲁棒性作为学习信号

**基本原理**：对同一输入施加不同扰动，要求模型给出一致的预测。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

model = SimpleClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

X_l, y_l, X_u = generate_data(n_labeled=30, n_unlabeled=200)

print('=== Consistency Regularization (UDA-style) ===')
for epoch in range(50):
    # Supervised loss on labeled data
    logits_l = model(X_l)
    sup_loss = F.cross_entropy(logits_l, y_l)
    
    # Consistency loss on unlabeled data
    noise1 = torch.randn_like(X_u) * 0.1
    noise2 = torch.randn_like(X_u) * 0.1
    logits_u1 = model(X_u + noise1)
    logits_u2 = model(X_u + noise2)
    
    # KL divergence between two perturbed predictions
    log_prob1 = F.log_softmax(logits_u1, dim=-1)
    prob2 = F.softmax(logits_u2, dim=-1)
    cons_loss = F.kl_div(log_prob1, prob2, reduction='batchmean')
    
    loss = sup_loss + cons_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        acc = (model(X_l).argmax(dim=-1) == y_l).float().mean()
        print(f'Epoch {epoch+1}: sup_loss={sup_loss.item():.4f}, cons_loss={cons_loss.item():.4f}, acc={acc:.3f}')

print('\nKey: Model should give consistent predictions under different perturbations of the same input.')

## 4. 协同训练 / 5. LLM辅助标注

**协同训练**：从不同视角互相提供伪标签

**LLM辅助标注**：用强模型为弱模型生成训练数据

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class ViewClassifier(nn.Module):
    def __init__(self, input_dim, hidden=32, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x):
        return self.net(x)

X_l, y_l, X_u = generate_data(n_labeled=20, n_unlabeled=200, dim=10)

# Split features into two views
X_l_v1, X_l_v2 = X_l[:, :5], X_l[:, 5:]
X_u_v1, X_u_v2 = X_u[:, :5], X_u[:, 5:]

model1 = ViewClassifier(5)
model2 = ViewClassifier(5)
opt1 = torch.optim.Adam(model1.parameters(), lr=1e-3)
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

print('=== Co-Training Demo ===')
for round_num in range(3):
    for epoch in range(20):
        loss1 = F.cross_entropy(model1(X_l_v1), y_l)
        loss2 = F.cross_entropy(model2(X_l_v2), y_l)
        opt1.zero_grad(); loss1.backward(); opt1.step()
        opt2.zero_grad(); loss2.backward(); opt2.step()
    
    with torch.no_grad():
        probs1 = F.softmax(model1(X_u_v1), dim=-1)
        probs2 = F.softmax(model2(X_u_v2), dim=-1)
        conf1, pred1 = probs1.max(dim=-1)
        conf2, pred2 = probs2.max(dim=-1)
    
    high_conf1 = conf1 > 0.9
    high_conf2 = conf2 > 0.9
    
    n_new1 = high_conf1.sum().item()
    n_new2 = high_conf2.sum().item()
    
    if n_new1 > 0:
        X_l_v1 = torch.cat([X_l_v1, X_u_v1[high_conf1]])
        X_l_v2 = torch.cat([X_l_v2, X_u_v2[high_conf1]])
        y_l = torch.cat([y_l, pred1[high_conf1]])
    if n_new2 > 0:
        X_l_v1 = torch.cat([X_l_v1, X_u_v1[high_conf2]])
        X_l_v2 = torch.cat([X_l_v2, X_u_v2[high_conf2]])
        y_l = torch.cat([y_l, pred2[high_conf2]])
    
    acc1 = (model1(X_l_v1[:20]).argmax(-1) == y_l[:20]).float().mean()
    print(f'Round {round_num+1}: labeled={y_l.shape[0]}, model1_added={n_new1}, model2_added={n_new2}, acc={acc1:.3f}')

print('\n=== LLM-Assisted Annotation ===')
print('Use GPT-4 to label unlabeled data -> train smaller model')
print('Example: Self-Instruct, Alpaca data generation pipeline')
print('Cost: ~$0.01-0.03 per GPT-4 label vs ~$0.50+ per human label')